In [1]:
# Notebook 1 — Celda 1: Configuración
import torch, os
from google.colab import drive
drive.mount('/content/drive')

BASE          = '/content/drive/MyDrive/nla_pipeline'
CHECKPOINT    = f'{BASE}/checkpoints/qwen_sujeto'
DIR_SALIDA    = f'{BASE}/activaciones'
CAPA          = 20   # extraemos de la capa 20 (de 28 totales)

# Detectar VRAM disponible y elegir modo automáticamente
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MODO_8BIT = vram_gb < 20   # True = Gratis (T4), False = Pro

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {vram_gb:.1f} GB')
print(f'Modo: {"8-bit (Colab Gratis)" if MODO_8BIT else "bfloat16 (Colab Pro)"}')



Mounted at /content/drive


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print('Cargando tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT, trust_remote_code=True
)

print(f'Cargando modelo en modo {"8-bit" if MODO_8BIT else "bfloat16"}...')
if MODO_8BIT:
    # Colab Gratis (T4): cuantización 8-bit → ~8-9 GB VRAM
    # Los pesos se comprimen a INT8, el forward sigue en FP16
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    modelo = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT,
        quantization_config=quantization_config, # Use BitsAndBytesConfig for 8-bit loading
        device_map='auto',
        trust_remote_code=True,
    )
else:
    # Colab Pro (L4/A100): bfloat16 completo → ~14-15 GB VRAM
    modelo = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT,
        torch_dtype=torch.bfloat16,
        device_map='cuda:0',
        trust_remote_code=True,
    )

modelo.eval()  # modo inferencia: desactiva dropout
print(f'✓ Modelo cargado — {modelo.config.num_hidden_layers} capas')

# Verificar VRAM usada
usado = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM usada: {usado:.1f} / {total:.1f} GB')

Cargando tokenizer...
Cargando modelo en modo 8-bit...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✓ Modelo cargado — 28 capas
VRAM usada: 8.7 / 15.6 GB


In [ ]:
# Notebook 1 — Celda 3: Hook y extracción
import numpy as np

# ── Registrar hook de extracción ───────────────────────
# Un hook es una función que PyTorch llama automáticamente
# justo después de que el bloque 20 termina su forward pass.
# Recibe (módulo, entrada, salida) y guarda la salida.
almacen = {}   # diccionario donde guardamos las activaciones

def hook_residual(modulo, entrada, salida):
    # salida: tensor [batch, seq_len, 3584]
    # .detach()  → desconecta del grafo de gradientes
    # .float()   → convierte a float32 para mejor precisión
    # .cpu()     → mueve a RAM (más seguro para guardar)
    almacen['capa_20'] = salida.detach().float().cpu()

# Registrar en el bloque 20 del modelo
handle = modelo.model.layers[CAPA].register_forward_hook(hook_residual)
print(f'✓ Hook registrado en layers[{CAPA}]')

# ── Texto de entrada ────────────────────────────────────
# Cambia este texto por el que quieras analizar
texto = 'The Eiffel Tower was built in Paris during the 1889 World Fair.'

# Formatear como conversación (Qwen usa chat template)
chat = [{'role': 'user', 'content': texto}]
input_ids = tokenizer.apply_chat_template(
    chat,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
)['input_ids'].to('cuda') # Access the 'input_ids' key to get the actual tensor

print(f'Tokens en la secuencia: {input_ids.shape[1]}')

# ── Forward pass ────────────────────────────────────────
# with torch.no_grad() evita calcular gradientes (ahorra VRAM y tiempo)
with torch.no_grad():
    _ = modelo(input_ids=input_ids)

# ── Recuperar activaciones ───────────────────────────────
tensor_act = almacen['capa_20']   # [1, T, 3584]
n_tokens   = tensor_act.shape[1]
print(f'Tensor capturado: {tensor_act.shape}  → [batch, tokens, d_model]')

# Calcular norma L2 para cada token (debe ser 80-200 para Qwen layer 20)
normas = tensor_act[0].norm(dim=-1)   # [T]
print('\n Norma L2 por token:')
for i, n in enumerate(normas):
    flag = '← ALTO (normal en primeros tokens)' if n > 1000 else ''
    print(f'  token[{i:2d}]: {n:.1f}  {flag}')

handle.remove()  # limpiar hook — buena práctica
print('\n✓ Extracción completada')

✓ Hook registrado en layers[20]
Tokens en la secuencia: 48


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Tensor capturado: torch.Size([1, 48, 3584])  → [batch, tokens, d_model]

 Norma L2 por token:
  token[ 0]: 282.6  
  token[ 1]: 165.7  
  token[ 2]: 13975.3  ← ALTO (normal en primeros tokens)
  token[ 3]: 129.4  
  token[ 4]: 114.9  
  token[ 5]: 100.5  
  token[ 6]: 109.1  
  token[ 7]: 107.5  
  token[ 8]: 100.0  
  token[ 9]: 112.8  
  token[10]: 113.4  
  token[11]: 109.5  
  token[12]: 106.5  
  token[13]: 112.8  
  token[14]: 107.4  
  token[15]: 107.3  
  token[16]: 118.0  
  token[17]: 110.6  
  token[18]: 100.1  
  token[19]: 155.7  
  token[20]: 105.9  
  token[21]: 106.1  
  token[22]: 105.3  
  token[23]: 114.3  
  token[24]: 114.2  
  token[25]: 113.4  
  token[26]: 134.4  
  token[27]: 122.0  
  token[28]: 131.9  
  token[29]: 125.3  
  token[30]: 127.4  
  token[31]: 114.0  
  token[32]: 125.5  
  token[33]: 120.0  
  token[34]: 109.7  
  token[35]: 107.8  
  token[36]: 87.2  
  token[37]: 113.6  
  token[38]: 103.9  
  token[39]: 110.3  
  token[40]: 125.7  
  token[41

In [ ]:
# Celda 4 — Guardar en Drive
# Notebook 1 — Celda 4: Guardar en Drive
import json

# Seleccionar posiciones a analizar (evitar las primeras 10)
inicio = 10
posiciones = list(range(inicio, n_tokens))
print(f'Guardando {len(posiciones)} tokens (pos {inicio} a {n_tokens-1})')

metadatos = {
    'texto': texto,
    'n_tokens': n_tokens,
    'posiciones': posiciones,
    'capa': CAPA,
    'd_model': 3584,
    'modo': '8bit' if MODO_8BIT else 'bfloat16',
}

# Guardar tensor completo de activaciones
ruta_npy  = f'{DIR_SALIDA}/activaciones_L{CAPA}.npy'
ruta_meta = f'{DIR_SALIDA}/metadatos.json'

np.save(ruta_npy, tensor_act[0].numpy())   # [T, 3584]
with open(ruta_meta, 'w') as f:
    json.dump(metadatos, f, ensure_ascii=False, indent=2)

print(f'✓ Activaciones guardadas: {ruta_npy}')
print(f'✓ Metadatos guardados:    {ruta_meta}')
print(f'\nTamaño del archivo: {os.path.getsize(ruta_npy)/1e6:.1f} MB')
print('\n🎉 Etapa 1 completada. Puedes cerrar este runtime.')
print('   Continúa con Notebook_2_av_verbalizar.ipynb')




Guardando 38 tokens (pos 10 a 47)
✓ Activaciones guardadas: /content/drive/MyDrive/nla_pipeline/activaciones/activaciones_L20.npy
✓ Metadatos guardados:    /content/drive/MyDrive/nla_pipeline/activaciones/metadatos.json

Tamaño del archivo: 0.7 MB

🎉 Etapa 1 completada. Puedes cerrar este runtime.
   Continúa con Notebook_2_av_verbalizar.ipynb
